In [1]:
import pandas as pd

In [3]:
#### TAKE OUT SCHOOLS WITH NO ELL STUDENTS

import pandas as pd
from pathlib import Path

in_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment.csv")
out_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonzero_ell.csv")

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

if "NUM_TOTAL_ELL" not in df.columns:
    raise ValueError(f"'NUM_TOTAL_ELL' not found. Columns: {list(df.columns)}")

df["NUM_TOTAL_ELL_NUM"] = pd.to_numeric(df["NUM_TOTAL_ELL"], errors="coerce").fillna(0)
filtered = df[df["NUM_TOTAL_ELL_NUM"] > 0].copy().drop(columns=["NUM_TOTAL_ELL_NUM"])

out_path.parent.mkdir(parents=True, exist_ok=True)  # safe even if folder already exists
filtered.to_csv(out_path, index=False)

print("Input rows:", len(df))
print("Output rows:", len(filtered))
print("Saved:", out_path)
filtered.head(10)

Input rows: 283
Output rows: 213
Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonzero_ell.csv


,ENTITY_NAME,PER_MALE,PER_FEMALE,PER_AM_IND,PER_BLACK,PER_HISP,PER_ASIAN,PER_WHITE,PER_MULTI,PER_KHALF,...,GRADE_12,GRADE_UGS,0-3 YEARS,4-6 YEARS,7+ YEARS,SIFE,ENG_NEW_LANG,ONE_TWO_WAY_DUAL_LANG,TRAN_BILING_ED,COUNTY
0,ALFRED-ALMOND CSD,0,100,0,0,0,100,0,0,0,...,1,0,-,-,-,-,-,-,-,ALLEGANY
1,ANDOVER CSD,100,0,0,0,100,0,0,0,0,...,0,0,-,-,-,-,-,-,-,ALLEGANY
6,FILLMORE CSD,0,100,0,0,100,0,0,0,0,...,0,0,-,-,-,-,-,-,-,ALLEGANY
7,WHITESVILLE CSD,67,33,0,0,100,0,0,0,0,...,0,0,-,-,-,-,-,-,-,ALLEGANY
8,CUBA-RUSHFORD CSD,100,0,0,0,0,0,100,0,0,...,0,0,-,-,-,-,-,-,-,ALLEGANY
10,WELLSVILLE CSD,50,50,0,0,25,75,0,0,0,...,0,0,-,-,-,-,-,-,-,ALLEGANY
12,CHENANGO FORKS CSD,73,27,0,0,0,0,100,0,0,...,1,0,9,2,2,0,13,0,0,BROOME
13,BINGHAMTON CITY SD,59,41,0,10,42,12,36,0,0,...,10,2,183,48,45,0,273,0,3,BROOME
14,HARPURSVILLE CSD,100,0,0,0,0,100,0,0,0,...,0,0,-,-,-,-,-,-,-,BROOME
15,SUSQUEHANNA VALLEY CSD,33,67,0,17,50,33,0,0,0,...,0,0,5,2,3,0,10,0,0,BROOME


In [4]:
import pandas as pd
from IPython.display import display

in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonzero_ell.csv"
out_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_sorted_most_ell.csv"

df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

# Convert to numeric
df["NUM_TOTAL_ELL"] = pd.to_numeric(df["NUM_TOTAL_ELL"], errors="coerce").fillna(0)

# If schools appear multiple times, sum by school
ranked = (
    df.groupby(["ENTITY_NAME", "INSTITUTION_ID", "COUNTY"], as_index=False)["NUM_TOTAL_ELL"]
      .sum()
      .sort_values("NUM_TOTAL_ELL", ascending=False)
)

ranked.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Total schools ranked:", len(ranked))
display(ranked.head(25))

Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_sorted_most_ell.csv
Total schools ranked: 213


,ENTITY_NAME,INSTITUTION_ID,COUNTY,NUM_TOTAL_ELL
126,NEWBURGH CITY SD,800000040250,ORANGE,1910
189,UTICA CITY SD,800000041284,ONEIDA,1439
114,MIDDLETOWN CITY SD,800000040414,ORANGE,1065
116,MONROE-WOODBURY CSD,800000040341,ORANGE,668
91,KINGSTON CITY SD,800000036308,ULSTER,585
162,SCHENECTADY CITY SD,800000038389,SCHENECTADY,433
49,FALLSBURG CSD,800000036680,SULLIVAN,365
97,LIBERTY CSD,800000036656,SULLIVAN,253
3,AMSTERDAM CITY SD,800000049983,MONTGOMERY,250
87,ITHACA CITY SD,800000036448,TOMPKINS,232


In [5]:
#### THIS REMOVES ANY COLLUNMS THAT ARE UNNEEDED FOR ANALYSIS

import pandas as pd
from IPython.display import display

# Load source data
in_path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment.csv"
df = pd.read_csv(in_path, dtype=str)
df.columns = df.columns.str.strip()

# Map requested output names -> possible source column names
col_map_options = {
    "ENTITY_NAME": ["ENTITY_NAME"],
    "NUM_TOTAL_ELL": ["NUM_TOTAL_ELL"],
    "NUM_HISP": ["NUM_HISP", "HISPANIC", "NUM_HISPANIC"],
    "0-3 YEARS": ["0-3 YEARS", "0_3 YEARS", "0-3_YEARS", "NUM_0_3_YEARS"],
    "4-6 YEARS": ["4-6 YEARS", "4_6 YEARS", "4-6_YEARS", "NUM_4_6_YEARS"],
    "7+ YEARS": ["7+ YEARS", "7_PLUS_YEARS", "7PLUS YEARS", "NUM_7_PLUS_YEARS"],
    "SIFE": ["SIFE", "NUM_SIFE"],
    "ENG_NEW_LANG": ["ENG_NEW_LANG", "ENGLISH_AS_A_NEW_LANGUAGE", "ENL"],
    "ONE_TWO_WAY_DUAL_LANG": ["ONE_TWO_WAY_DUAL_LANG", "ONE_TWO_WAY_DUAL_LANGUAGE"],
    "TRAN_BILING_ED": ["TRAN_BILING_ED", "TRANSITIONAL_BILINGUAL_ED", "TBE"],
    "COUNTY": ["COUNTY"]
}

# Resolve actual columns present in your file
resolved = {}
missing = []
for out_col, options in col_map_options.items():
    found = next((c for c in options if c in df.columns), None)
    if found is None:
        missing.append(out_col)
    else:
        resolved[out_col] = found

if missing:
    raise ValueError(f"Missing required columns in file: {missing}\nAvailable: {list(df.columns)}")

# Build filtered dataset with requested output column names
new_df = df[[resolved[c] for c in col_map_options.keys()]].rename(
    columns={resolved[k]: k for k in col_map_options.keys()}
)

# Remove schools with no ELLs and sort most -> least
new_df["NUM_TOTAL_ELL"] = pd.to_numeric(new_df["NUM_TOTAL_ELL"], errors="coerce").fillna(0)
new_df = new_df[new_df["NUM_TOTAL_ELL"] > 0].sort_values("NUM_TOTAL_ELL", ascending=False).reset_index(drop=True)

print("Rows kept:", len(new_df))
display(new_df.head(20))

Rows kept: 213


,ENTITY_NAME,NUM_TOTAL_ELL,NUM_HISP,0-3 YEARS,4-6 YEARS,7+ YEARS,SIFE,ENG_NEW_LANG,ONE_TWO_WAY_DUAL_LANG,TRAN_BILING_ED,COUNTY
0,NEWBURGH CITY SD,1910,1818,1277,484,465,45,1241,151,834,ORANGE
1,UTICA CITY SD,1439,318,1016,397,369,137,1757,25,0,ONEIDA
2,MIDDLETOWN CITY SD,1065,997,913,292,176,20,496,430,453,ORANGE
3,MONROE-WOODBURY CSD,668,596,617,186,95,80,898,0,0,ORANGE
4,KINGSTON CITY SD,585,539,409,178,85,48,669,0,0,ULSTER
5,SCHENECTADY CITY SD,433,209,331,99,98,7,527,0,0,SCHENECTADY
6,FALLSBURG CSD,365,350,268,124,63,8,356,99,0,SULLIVAN
7,LIBERTY CSD,253,235,223,60,38,11,311,0,0,SULLIVAN
8,AMSTERDAM CITY SD,250,233,136,79,77,0,292,0,0,MONTGOMERY
9,ITHACA CITY SD,232,41,222,42,18,0,281,0,1,TOMPKINS


In [7]:
#### DOWNLOAD THE DATA SET

from IPython.display import FileLink, display

# Assumes your cleaned dataframe is named `new_df` from the previous cell
filename = "enrollement_rural_cleaned.csv"  # using your requested filename

new_df.to_csv(filename, index=False)
print(f"Saved: {filename}")

# Click this link in Jupyter to download
display(FileLink(filename))

Saved: enrollement_rural_cleaned.csv


/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollement_rural_cleaned.csv

In [9]:
import pandas as pd
from pathlib import Path

enroll_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollement_rural_cleaned.csv")
grad2_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv")
out_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_rural_cleaned_grad2_only.csv")

enroll = pd.read_csv(enroll_path, dtype=str)
grad2 = pd.read_csv(grad2_path, dtype=str)

enroll.columns = enroll.columns.str.strip()
grad2.columns = grad2.columns.str.strip()

enroll["_name_key"] = enroll["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()
grad2["_name_key"] = grad2["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()

filtered = enroll[enroll["_name_key"].isin(set(grad2["_name_key"]))].copy()
filtered = filtered.drop(columns=["_name_key"])

out_path.parent.mkdir(parents=True, exist_ok=True)  # ensures folder exists
filtered.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Rows kept:", len(filtered))
display(filtered.head(20))

Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_rural_cleaned_grad2_only.csv
Rows kept: 189


,ENTITY_NAME,NUM_TOTAL_ELL,NUM_HISP,0-3 YEARS,4-6 YEARS,7+ YEARS,SIFE,ENG_NEW_LANG,ONE_TWO_WAY_DUAL_LANG,TRAN_BILING_ED,COUNTY
15,HUDSON CITY SD,148,88,106,47,28,1,178,0,0,COLUMBIA
16,INDIAN RIVER CSD,132,79,151,20,2,0,173,0,0,JEFFERSON
18,PINE BUSH CSD,119,90,88,53,28,0,169,0,0,ORANGE
19,VESTAL CSD,118,6,97,26,11,0,134,0,0,BROOME
21,UNION-ENDICOTT CSD,114,55,122,28,13,1,161,0,0,BROOME
24,KINDERHOOK CSD,96,83,67,25,23,0,115,0,0,COLUMBIA
25,ROME CITY SD,94,57,63,24,20,0,106,0,1,ONEIDA
26,WALLKILL CSD,85,82,67,17,21,1,105,0,0,ULSTER
27,MINISINK VALLEY CSD,85,59,48,39,14,0,101,0,0,ORANGE
28,CORNWALL CSD,73,39,61,16,10,0,83,0,4,ORANGE


In [12]:
import pandas as pd
from pathlib import Path

rural_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollement_rural_cleaned.csv")
grad2_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/grad2.csv")
nonrural_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_cleaned.csv")
out_path = Path("/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_plus_removed.csv")

rural = pd.read_csv(rural_path, dtype=str)
grad2 = pd.read_csv(grad2_path, dtype=str)
nonrural = pd.read_csv(nonrural_path, dtype=str)

for d in (rural, grad2, nonrural):
    d.columns = d.columns.str.strip()

# Name keys for matching
rural["_name_key"] = rural["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()
grad2["_name_key"] = grad2["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()
nonrural["_name_key"] = nonrural["ENTITY_NAME"].fillna("").astype(str).str.strip().str.upper()

# These are the rows previously taken off rural (not in grad2)
removed_from_rural = rural[~rural["_name_key"].isin(set(grad2["_name_key"]))].copy()

# Align columns to nonrural (so concat works cleanly)
cols = [c for c in nonrural.columns if c in removed_from_rural.columns]
removed_from_rural = removed_from_rural[cols].copy()

# ADD them to nonrural (no dedupe/removal)
updated = pd.concat([nonrural, removed_from_rural], ignore_index=True)

updated.to_csv(out_path, index=False)

print("Rows in nonrural before:", len(nonrural))
print("Rows taken off rural and added:", len(removed_from_rural))
print("Rows in output:", len(updated))
print("Saved:", out_path)
display(removed_from_rural.head(20))

Rows in nonrural before: 943
Rows taken off rural and added: 24
Rows in output: 967
Saved: /Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_plus_removed.csv


,ENTITY_NAME,NUM_TOTAL_ELL,NUM_HISP,0-3 YEARS,4-6 YEARS,7+ YEARS,SIFE,ENG_NEW_LANG,ONE_TWO_WAY_DUAL_LANG,TRAN_BILING_ED,COUNTY,_name_key
0,NEWBURGH CITY SD,1910,1818,1277,484,465,45,1241,151,834,ORANGE,NEWBURGH CITY SD
1,UTICA CITY SD,1439,318,1016,397,369,137,1757,25,0,ONEIDA,UTICA CITY SD
2,MIDDLETOWN CITY SD,1065,997,913,292,176,20,496,430,453,ORANGE,MIDDLETOWN CITY SD
3,MONROE-WOODBURY CSD,668,596,617,186,95,80,898,0,0,ORANGE,MONROE-WOODBURY CSD
4,KINGSTON CITY SD,585,539,409,178,85,48,669,0,0,ULSTER,KINGSTON CITY SD
5,SCHENECTADY CITY SD,433,209,331,99,98,7,527,0,0,SCHENECTADY,SCHENECTADY CITY SD
6,FALLSBURG CSD,365,350,268,124,63,8,356,99,0,SULLIVAN,FALLSBURG CSD
7,LIBERTY CSD,253,235,223,60,38,11,311,0,0,SULLIVAN,LIBERTY CSD
8,AMSTERDAM CITY SD,250,233,136,79,77,0,292,0,0,MONTGOMERY,AMSTERDAM CITY SD
9,ITHACA CITY SD,232,41,222,42,18,0,281,0,1,TOMPKINS,ITHACA CITY SD


In [13]:
import pandas as pd

path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_rural_cleaned_grad2_only.csv"
df = pd.read_csv(path, dtype=str)
df.columns = df.columns.str.strip()

# Convert columns to numeric (treat "-" or blanks as missing)
df["7+ YEARS"] = pd.to_numeric(df["7+ YEARS"].replace("-", pd.NA), errors="coerce")
df["NUM_TOTAL_ELL"] = pd.to_numeric(df["NUM_TOTAL_ELL"].replace("-", pd.NA), errors="coerce")

sum_7plus = df["7+ YEARS"].sum(skipna=True)
sum_total_ell = df["NUM_TOTAL_ELL"].sum(skipna=True)

ratio = sum_7plus / sum_total_ell if sum_total_ell > 0 else pd.NA
percent = ratio * 100 if pd.notna(ratio) else pd.NA

print("Sum of 7+ YEARS:", sum_7plus)
print("Sum of NUM_TOTAL_ELL:", sum_total_ell)
print("7+ YEARS / NUM_TOTAL_ELL:", ratio)
print("Percent:", round(percent, 2), "%")

Sum of 7+ YEARS: 466.0
Sum of NUM_TOTAL_ELL: 3238
7+ YEARS / NUM_TOTAL_ELL: 0.1439159975293391
Percent: 14.39 %


In [14]:
import pandas as pd

path = "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_updated.csv"
df = pd.read_csv(path, dtype=str)
df.columns = df.columns.str.strip()

# Convert to numeric (treat "-" or blanks as missing)
df["7+ YEARS"] = pd.to_numeric(df["7+ YEARS"].replace("-", pd.NA), errors="coerce")
df["NUM_TOTAL_ELL"] = pd.to_numeric(df["NUM_TOTAL_ELL"].replace("-", pd.NA), errors="coerce")

sum_7plus = df["7+ YEARS"].sum(skipna=True)
sum_total_ell = df["NUM_TOTAL_ELL"].sum(skipna=True)

ratio = sum_7plus / sum_total_ell if sum_total_ell > 0 else pd.NA
percent = ratio * 100 if pd.notna(ratio) else pd.NA

print("Sum of 7+ YEARS:", sum_7plus)
print("Sum of NUM_TOTAL_ELL:", sum_total_ell)
print("7+ YEARS / NUM_TOTAL_ELL:", ratio)
print("Percent:", round(percent, 2), "%")

Sum of 7+ YEARS: 48542.0
Sum of NUM_TOTAL_ELL: 259829
7+ YEARS / NUM_TOTAL_ELL: 0.1868228719657929
Percent: 18.68 %


In [15]:
import pandas as pd

files = {
    "rural": "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_rural_cleaned_grad2_only.csv",
    "nonrural": "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_updated.csv",
}

for label, path in files.items():
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()

    # Convert to numeric (treat "-" or blanks as missing)
    df["SIFE"] = pd.to_numeric(df["SIFE"].replace("-", pd.NA), errors="coerce")
    df["NUM_TOTAL_ELL"] = pd.to_numeric(df["NUM_TOTAL_ELL"].replace("-", pd.NA), errors="coerce")

    sum_sife = df["SIFE"].sum(skipna=True)
    sum_total_ell = df["NUM_TOTAL_ELL"].sum(skipna=True)

    ratio = sum_sife / sum_total_ell if sum_total_ell > 0 else pd.NA
    percent = ratio * 100 if pd.notna(ratio) else pd.NA

    print(f"\n=== {label.upper()} ===")
    print("Sum of SIFE:", sum_sife)
    print("Sum of NUM_TOTAL_ELL:", sum_total_ell)
    print("SIFE / NUM_TOTAL_ELL:", ratio)
    print("Percent:", round(percent, 2), "%")


=== RURAL ===
Sum of SIFE: 34.0
Sum of NUM_TOTAL_ELL: 3238
SIFE / NUM_TOTAL_ELL: 0.010500308832612723
Percent: 1.05 %

=== NONRURAL ===
Sum of SIFE: 13531.0
Sum of NUM_TOTAL_ELL: 259829
SIFE / NUM_TOTAL_ELL: 0.05207655804394429
Percent: 5.21 %


In [16]:
import pandas as pd

files = {
    "rural": "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_rural_cleaned_grad2_only.csv",
    "nonrural": "/Users/emmamurphy/dataProjects/migranted/Data/Cleaned data and Analysis/enrollment_nonrural_updated.csv",
}

metrics = ["0-3 YEARS", "4-6 YEARS", "7+ YEARS", "SIFE"]

tables = []

for dataset, path in files.items():
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()

    # numeric conversion
    df["NUM_TOTAL_ELL"] = pd.to_numeric(df["NUM_TOTAL_ELL"].replace("-", pd.NA), errors="coerce")
    total_ell = df["NUM_TOTAL_ELL"].sum(skipna=True)

    rows = []
    for m in metrics:
        df[m] = pd.to_numeric(df[m].replace("-", pd.NA), errors="coerce")
        metric_sum = df[m].sum(skipna=True)
        ratio = metric_sum / total_ell if total_ell > 0 else pd.NA
        percent = ratio * 100 if pd.notna(ratio) else pd.NA

        rows.append({
            "dataset": dataset,
            "metric": m,
            "sum_value": metric_sum,
            "sum_num_total_ell": total_ell,
            "ratio_to_num_total_ell": ratio,
            "percent_of_num_total_ell": round(percent, 2) if pd.notna(percent) else pd.NA
        })

    tables.append(pd.DataFrame(rows))

summary_table = pd.concat(tables, ignore_index=True)

# long-format table
display(summary_table)

# optional wide-format table
wide = summary_table.pivot(index="metric", columns="dataset", values="percent_of_num_total_ell")
display(wide)

,dataset,metric,sum_value,sum_num_total_ell,ratio_to_num_total_ell,percent_of_num_total_ell
0,rural,0-3 YEARS,2609.0,3238,0.805744,80.57
1,rural,4-6 YEARS,802.0,3238,0.247684,24.77
2,rural,7+ YEARS,466.0,3238,0.143916,14.39
3,rural,SIFE,34.0,3238,0.010500,1.05
4,nonrural,0-3 YEARS,224088.0,259829,0.862444,86.24
5,nonrural,4-6 YEARS,61820.0,259829,0.237926,23.79
6,nonrural,7+ YEARS,48542.0,259829,0.186823,18.68
7,nonrural,SIFE,13531.0,259829,0.052077,5.21


dataset,nonrural,rural
metric,,
0-3 YEARS,86.24,80.57
4-6 YEARS,23.79,24.77
7+ YEARS,18.68,14.39
SIFE,5.21,1.05
